In [1]:
from datasets import load_dataset

ds = load_dataset("phunc20/nj_biergarten_captcha")

c:\Users\Kristoffer\Documents\Dev\UNI\TAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Print the key of the first example in the training set
print(ds["train"][0]["__key__"])
#Show the image
ds["train"][0]["jpg"].show()
# Extract the label by extracting characters after '_'
print(*ds["train"][0]["__key__"].split("_")[1:])
# Print size of dataset
print(len(ds["train"]))
# Print shape of image
print(ds["train"][0]["jpg"].size)

001/2024-10-08T18:21:10.578994_y2jhuv
y2jhuv
476118
(140, 50)


In [3]:
import numpy as np
import torch
import cv2
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter


* Functions

In [4]:
def char_to_1_hot(char, sorted_chars):
    # Convert a character to a one-hot encoded vector
    index = sorted_chars.index(char)
    one_hot = np.zeros(len(sorted_chars), dtype=np.uint8)
    one_hot[index] = 1.0
    return one_hot

def one_hot(label, sorted_chars):
    return np.hstack([char_to_1_hot(char, sorted_chars) for char in label[0]])

def one_hot_to_char(x, sorted_chars):
    y = np.array(x)
    y = y.squeeze()
    assert len(y) == len(sorted_chars)
    idx = np.argmax(y)
    return(sorted_chars[idx])

def one_hot_to_label(x, sorted_chars, char_per_label):
    y = np.array(x)
    y = y.squeeze()
    label_list = []
    assert len(y) == (len(sorted_chars * char_per_label))
    for i in range(0, char_per_label):
        start = i * len(sorted_chars)
        end = start + len(sorted_chars)
        label_list.append(one_hot_to_char(y[start:end], sorted_chars))
    return "".join(label_list)

In [5]:
def extract_labels(dataset):
    # Extract the labels from the dataset
    val = np.array(dataset.split("_")[1:])
    return val
    #*ds["train"][0]["__key__"].split("_")[1:]
class CustomCaptchaDataset(Dataset):
    def __init__(self, transform=None, sorted_chars=None):
        self.transform = transform
        self.sorted_chars = sorted_chars

    def __len__(self):
        return len(ds["train"])

    def __getitem__(self, idx):

        label = one_hot(extract_labels(ds["train"][idx]["__key__"]), self.sorted_chars)
        image = np.array(ds["train"][idx]["jpg"])

        if self.transform:
            image = self.transform(image)

        return image, label
    
    def get_labels(self, range):
        label = extract_labels(ds["train"][range]["__key__"])
        return label

In [6]:
data_points = 476118
batch_size = 128
char_per_label = 6

sorted_chars = ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

train_size = int(data_points * 0.75)
test_size = int(data_points - train_size)
print(f"Train size: {train_size}, Test size: {test_size}")

# Grayscale the data
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Resize((25, 70)),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=0.1)
])

dataset = CustomCaptchaDataset(transform=transform, sorted_chars=sorted_chars)

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders for batch processing\n",
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(dataset[0][0].shape)
print(dataset[0][1])

Train size: 357088, Test size: 119030
torch.Size([1, 25, 70])
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]


In [7]:
# Get all of the characters in the dataset
sorted_chars = ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
test_string = "1abdg2"

print(len(sorted_chars))
print(one_hot(test_string, sorted_chars))
print(one_hot_to_label(dataset[0][1], sorted_chars, char_per_label))

35
[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
y2jhuv


In [8]:
class GlobalAttention(nn.Module):
    def __init__(self, num_channels):
        super(GlobalAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Conv2d(num_channels, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights

In [9]:
class ModelWithAttention(nn.Module):
    def __init__(self, num_characters):
        super(ModelWithAttention, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=(3,3), padding='same')        # 64, 50, 140
        self.bn1 = nn.BatchNorm2d(64)                                           
        self.pool = nn.MaxPool2d(kernel_size=(2,2))                             # 64, 49, 139

        self.conv2 = nn.Conv2d(64, 128, kernel_size=(3,3), padding='same')      # 128, 49, 139
        self.bn2 = nn.BatchNorm2d(128)

        self.conv3 = nn.Conv2d(128, 256, kernel_size=(3,3), padding='same')
        self.bn3 = nn.BatchNorm2d(256)

        self.conv4 = nn.Conv2d(256, 512, kernel_size=(3,3), padding='same')
        self.bn4 = nn.BatchNorm2d(512)
        self.pool2 = nn.MaxPool2d(kernel_size=(1,2))                            # 512, 6, 8   

        # Global attention layer
        self.attention = GlobalAttention(512)

        # Fully connected layers
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(512 * 1 * 6 * 8, 512)
        self.bn5 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(0.5)

        self.fc2 = nn.Linear(512, 512)
        self.bn6 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(0.75)

        self.output = nn.Linear(512, num_characters)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = nn.ReLU()(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = nn.ReLU()(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = nn.ReLU()(x)
        x = self.pool(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = nn.ReLU()(x)
        x = self.pool2(x)

        # Apply global attention
        x = self.attention(x)

        # Flatten the output
        x = self.flatten(x)

        # Fully connected layers
        x = self.fc1(x)
        x = self.bn5(x)
        x = nn.ReLU()(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.bn6(x)
        x = nn.ReLU()(x)
        x = self.dropout2(x)

        # Output layer
        x = self.output(x)

        return x
    
class CNNModel(nn.Module):
    def __init__(self, alphabet_size):
        super(CNNModel, self,).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(5,5), padding='same')
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=(2,2))

        self.conv2 = nn.Conv2d(32, 48, kernel_size=(5,5), padding='same')
        self.bn2 = nn.BatchNorm2d(48)
        self.pool2 = nn.MaxPool2d(kernel_size=(2,2))

        self.conv3 = nn.Conv2d(48, 64, kernel_size=(5,5), padding='same')
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=(2,2))

        self.attention = GlobalAttention(64)

        self.flatten = nn.Flatten()
        self.fc1 = nn.LazyLinear(512)

        # Softmax layer
        self.softmax_layer_1 = nn.Linear(512, alphabet_size)
        self.softmax_1 = nn.Softmax(dim=1)
        self.softmax_layer_2 = nn.Linear(512, alphabet_size)
        self.softmax_2 = nn.Softmax(dim=1)
        self.softmax_layer_3 = nn.Linear(512, alphabet_size)
        self.softmax_3 = nn.Softmax(dim=1)
        self.softmax_layer_4 = nn.Linear(512, alphabet_size)
        self.softmax_4 = nn.Softmax(dim=1)
        self.softmax_layer_5 = nn.Linear(512, alphabet_size)
        self.softmax_5 = nn.Softmax(dim=1)
        self.softmax_layer_6 = nn.Linear(512, alphabet_size)
        self.softmax_6 = nn.Softmax(dim=1)

        # Dropout layer
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)

        # Apply global attention
        x = self.attention(x)

        # Flatten the output
        x = self.flatten(x)

        # Fully connected layers
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Apply softmax to each character position
        char_1 = self.softmax_layer_1(x)
        char_2 = self.softmax_layer_2(x)
        char_3 = self.softmax_layer_3(x)
        char_4 = self.softmax_layer_4(x)
        char_5 = self.softmax_layer_5(x)
        char_6 = self.softmax_layer_6(x)

        char_1 = self.softmax_1(char_1)
        char_2 = self.softmax_2(char_2)
        char_3 = self.softmax_3(char_3)
        char_4 = self.softmax_4(char_4)
        char_5 = self.softmax_5(char_5)
        char_6 = self.softmax_6(char_6)

        # Combine into x
        x = torch.cat((char_1, char_2, char_3, char_4, char_5, char_6), dim=1)

        #return char_1, char_2, char_3, char_4, char_5, char_6
        return x
    

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

#model = ModelWithAttention(len(sorted_chars) * char_per_label).to(device)
model = CNNModel(len(sorted_chars)).to(device)

# Test the model with the first image from train_loader
image, label = next(iter(train_loader))
image = image.to(device)
label = label.to(device)
output = model(image)
print("Parameters in model: ", sum(p.numel() for p in model.parameters() if p.requires_grad))
model.eval()

Using device:  cuda
Parameters in model:  1011173


CNNModel(
  (conv1): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1), padding=same)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 48, kernel_size=(5, 5), stride=(1, 1), padding=same)
  (bn2): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(48, 64, kernel_size=(5, 5), stride=(1, 1), padding=same)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (attention): GlobalAttention(
    (attention): Sequential(
      (0): Conv2d(64, 1, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [ ]:
total_time = 0
max_epochs = 20
val_interval = 1
step = 0
best_val_loss = float('inf')
trainingEpoch_loss = []
trainStepsLoss = []
validationEpoch_loss = []
print_interval = 1

writer = SummaryWriter()

#model=ModelWithAttention(len(sorted_chars) * char_per_label).to(device)
model = CNNModel(len(sorted_chars)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), 1e-4, amsgrad = True)

total_batches = 0

for epoch in range(max_epochs):
    model.train()
    print("-"*30)
    print(f"epoch {epoch + 1}/{max_epochs}")
    print("-"*30)
    model.train()
    epoch_loss = 0.0
    step = 0

    total_train = 0
    total_val = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data
        inputs = inputs.to(device)
        labels = labels.to(device)
        labels = labels.to(torch.float32)

        # Normal pipeline
        optimizer.zero_grad()
        outputs = model(inputs)
        #loss = calc_loss(outputs, labels, char_per_label, len(sorted_chars))
        loss = F.binary_cross_entropy(outputs, labels, reduction='mean')
        loss.backward()
        optimizer.step()

        # Calculate accuracy
        correct = 0
        fail = 0
        for i in range(len(labels)):
            pred = one_hot_to_label(outputs[i].cpu().detach().numpy(), sorted_chars, char_per_label)
            true = one_hot_to_label(labels[i].cpu().detach().numpy(), sorted_chars, char_per_label)
            if pred == true:
                correct += 1
            else:
                fail += 1
        total_train += correct

        epoch_loss += loss.item()
        #if (epoch) % print_interval == 0:
        #    print(f"{step}/{train_size // train_loader.batch_size}, " f"train_loss: {loss.item():.4f}")

        trainStepsLoss.append(loss.item())

        total_batches += 1
        writer.add_scalar("Total batches", epoch_loss/step, total_batches)

    epoch_loss /= step
    trainingEpoch_loss.append(epoch_loss)

    writer.add_scalar("Loss/train", epoch_loss, epoch)
    writer.add_scalar("Accuracy/train", total_train / len(train_loader.dataset) * 100, epoch)
    
    # Validation part
    if epoch % val_interval == 0:
        model.eval()  # Set the model to evaluation mode
        val_loss = 0
        val_steps = 0
                        
        with torch.no_grad():  # Disable gradient calculation
            for batch_data in test_loader:  # Assuming you have a val_dataloader

                inputs, labels = batch_data
                inputs = inputs.to(device)
                labels = labels.to(device)
                labels = labels.to(torch.float32)
                
                outputs = model(inputs)
                #loss = calc_loss(outputs, labels, char_per_label, len(sorted_chars))
                loss = F.binary_cross_entropy(outputs, labels, reduction='mean')
                val_loss += loss.item()
                val_steps += 1

                # Calculate accuracy
                correct = 0
                fail = 0
                for i in range(len(labels)):
                    pred = one_hot_to_label(outputs[i].cpu().detach().numpy(), sorted_chars, char_per_label)
                    true = one_hot_to_label(labels[i].cpu().detach().numpy(), sorted_chars, char_per_label)
                    if pred == true:
                        correct += 1
                    else:
                        fail += 1
                total_val += correct

        
        average_val_loss = val_loss / val_steps
        validationEpoch_loss.append(average_val_loss)
        
        # Checkpointing
        if average_val_loss < best_val_loss:
            best_val_loss = average_val_loss
            torch.save(model.state_dict(), 'model_best.pth')
            #print(f"Epoch {epoch}: New best model saved with val_loss: {average_val_loss}")
        
        # Optionally, print the validation loss
        print(f"   Avg. train loss: {epoch_loss:.4f}")
        print(f"   Train Acc. = {total_train / len(train_loader.dataset) * 100:.4f}%")
        print(f"   Avg. val. loss = {average_val_loss:.4f}")
        print(f"   Val. Acc. = {total_val / len(test_loader.dataset) * 100:.4f}%")

        writer.add_scalar("Loss/val", average_val_loss, epoch)
        writer.add_scalar("Accuracy/val", total_val / len(test_loader.dataset) * 100, epoch)

writer.flush()

------------------------------
epoch 1/20
------------------------------


KeyboardInterrupt: 